# WEMoE NLP: Colab pipeline

Runs the full reproduction on a Colab GPU (a T4 is enough):

1. Setup: Drive, clone, install, unit tests
2. Optional smoke run (a few minutes, numbers are meaningless)
3. Experiment 1: five single-task BERT experts
4. Baselines: Simple Averaging and Task Arithmetic (lambda chosen on dev)
5. Experiment 2: WEMoE + FSFT (3 seeds), k sweep, intermediate-only ablation
6. Experiment 4: parameters, storage, latency
7. Summary tables

Checkpoints and results are written to Google Drive. Every step skips work that is already done,
so after a disconnect just rerun the cells.

If the repository is private, add a Colab secret named `GITHUB_TOKEN` (fine-grained, read-only).
Never paste a token into a cell.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/wemoe-nlp'
PATHS = f'--set paths.checkpoint_dir={DRIVE_ROOT}/checkpoints --set paths.results_dir={DRIVE_ROOT}/results'
CFG = f'--config configs/default.yaml {PATHS}'

Mounted at /content/drive


In [3]:
import os
import subprocess

REPO = 'nguyenbaanh7779/wemoe-nlp'
BRANCH = 'feat/wemoe-fsft-pipeline'
WORKDIR = '/content/wemoe-nlp'

if not os.path.exists(WORKDIR):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')  # only needed for a private repository
    except Exception:
        pass
    url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
    result = subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, url, WORKDIR],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError('git clone failed: ' + result.stderr.replace(token or '<no token>', '***'))
    # keep the token out of .git/config
    subprocess.run(['git', '-C', WORKDIR, 'remote', 'set-url', 'origin', f'https://github.com/{REPO}.git'],
                   check=True)

%cd {WORKDIR}
!git log --oneline -1
!pip install -q -r requirements.txt

/content/wemoe-nlp
35f8552 (HEAD -> feat/wemoe-fsft-pipeline, origin/feat/wemoe-fsft-pipeline) Feat: WEMoE + FSFT pipeline for BERT on five GLUE tasks


In [4]:
!python -m pytest -q

.........................................                                [100%]
41 passed in 1.92s


## Optional: smoke run

Tiny subsets and a few steps, to check the whole pipeline end to end. Outputs stay on the VM.

In [5]:
SMOKE = '--config configs/smoke.yaml --set paths.checkpoint_dir=/content/smoke/checkpoints --set paths.results_dir=/content/smoke/results'
!python scripts/train_expert.py {SMOKE}
!python scripts/eval_baselines.py {SMOKE}
!python scripts/train_wemoe.py {SMOKE}
!python scripts/summarize_results.py --results-dir /content/smoke/results

[18:02:26] INFO httpx: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
[18:02:26] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[18:02:26] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
[18:02:26] INFO httpx: HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/86b5e0934494bd15c9632b12f734a8a67f723594/config.json "HTTP/1.1 200 OK"
[18:02:26] WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[18:02:26] INFO httpx: HTTP Request: GET https://huggingface.co/bert-base-uncased/resolve/86b5e0934494bd15c9632b12f734a8a67f723594/config.json "HTTP/1.1 200 OK"
config.json: 100% 570/570 [00:00<00:00, 2.29MB/s]
[18:02:26] INFO httpx: HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolv

## Experiment 1: single-task experts

One recipe for every task (lr 2e-5, batch 32, 3 epochs, max length 128, fp16). The best epoch is
chosen on the dev split carved from the training set. About 40-60 minutes on a T4.

In [6]:
!python scripts/train_expert.py {CFG}

[18:04:23] INFO root: Skipping mrpc: already trained (use --overwrite to retrain)
[18:04:23] INFO root: Skipping rte: already trained (use --overwrite to retrain)
[18:04:23] INFO root: Skipping cola: already trained (use --overwrite to retrain)
[18:04:23] INFO root: Skipping sst2: already trained (use --overwrite to retrain)
[18:04:23] INFO root: Skipping qnli: already trained (use --overwrite to retrain)


## Baselines: Simple Averaging and Task Arithmetic

Task Arithmetic lambda is chosen on dev. WEMoE uses the same lambda to initialise its routers.

In [7]:
!python scripts/eval_baselines.py {CFG}

[18:04:33] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[18:04:34] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
[18:04:34] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
[18:04:34] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
[18:04:34] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
[18:04:34] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
[18:04:34] INFO datasets: TensorFlow 

## Experiment 2: WEMoE + FSFT

Routers only, cross-entropy on k=256 labelled examples per task from the few-shot pool (never seen by
the experts), best step on dev, a single evaluation on the official validation split.

In [8]:
for seed in [0, 1, 2]:
    !python scripts/train_wemoe.py {CFG} --set wemoe.seed={seed}

[18:06:30] INFO root: Skipping full-mean-k256-s0: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k256-s0 exists (use --overwrite to rerun)
[18:06:34] INFO root: Skipping full-mean-k256-s1: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k256-s1 exists (use --overwrite to rerun)
[18:06:38] INFO root: Skipping full-mean-k256-s2: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k256-s2 exists (use --overwrite to rerun)


## k sweep (Experiment 6) and MoE-scope ablation (intermediate only, as in the DANC code)

In [9]:
for k in [16, 64]:
    for seed in [0, 1, 2]:
        !python scripts/train_wemoe.py {CFG} --set wemoe.k={k} --set wemoe.seed={seed}

for seed in [0, 1, 2]:
    !python scripts/train_wemoe.py {CFG} --set wemoe.moe_scope=intermediate --set wemoe.seed={seed}

[18:06:42] INFO root: Skipping full-mean-k16-s0: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k16-s0 exists (use --overwrite to rerun)
[18:06:46] INFO root: Skipping full-mean-k16-s1: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k16-s1 exists (use --overwrite to rerun)
[18:06:50] INFO root: Skipping full-mean-k16-s2: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k16-s2 exists (use --overwrite to rerun)
[18:06:54] INFO root: Skipping full-mean-k64-s0: /content/drive/MyDrive/wemoe-nlp/results/wemoe/full-mean-k64-s0 exists (use --overwrite to rerun)
[18:07:04] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[18:07:04] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
[18:07:04] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTT

## Experiment 4: parameters, storage and latency

In [10]:
!python scripts/count_params.py {CFG} --latency

[19:02:08] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased "HTTP/1.1 307 Temporary Redirect"
[19:02:08] INFO httpx: HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased "HTTP/1.1 200 OK"
Loading weights: 100% 199/199 [00:00<00:00, 4767.53it/s]
| Deployment | # checkpoints | Stored params | Size (fp32) | vs 5 models |
|---|---:|---:|---:|---:|
| 5 single-task models | 5 | 547.4M | 2.04 GiB | 100.0% |
| Task Arithmetic | 1 | 109.5M | 0.41 GiB | 20.0% |
| WEMoE (full MLP) | 1 | 400.0M | 1.49 GiB | 73.1% |
| WEMoE (intermediate only) | 1 | 258.4M | 0.96 GiB | 47.2% |

Saved /content/drive/MyDrive/wemoe-nlp/results/efficiency.json


## Summary

In [11]:
!python scripts/summarize_results.py --results-dir {DRIVE_ROOT}/results

# Results (official GLUE validation splits)

| Model | MRPC (acc/F1) | RTE (acc) | CoLA (MCC) | SST-2 (acc) | QNLI (acc) | Avg score | Normalized |
|---|---:|---:|---:|---:|---:|---:|---:|
| Single-task experts | 82.1/88.1 | 63.2 | 59.1 | 92.2 | 91.2 | 78.2 | 100.0 |
| Simple Averaging (λ=1/5) | 68.4/81.2 | 46.9 | 23.3 | 55.4 | 65.9 | 53.3 | 66.8 |
| Task Arithmetic (λ=0.3) | 68.6/81.3 | 47.3 | 16.9 | 69.8 | 77.9 | 57.4 | 70.6 |
| WEMoE init (full, λ=0.3) | 68.5±0.1/81.3±0.1 | 47.1±0.2 | 19.3±2.0 | 70.2±1.1 | 77.7±0.2 | 57.8±0.6 | 71.3±1.0 |
| WEMoE + FSFT (full, mean, k=16, seeds=[0, 1, 2]) | 68.7±2.3/80.6±2.2 | 48.0±1.0 | 27.6±14.1 | 79.6±8.1 | 74.0±6.5 | 60.8±2.7 | 75.6±4.6 |
| WEMoE init (full, λ=0.3) | 68.5±0.1/81.3±0.1 | 47.1±0.2 | 19.3±2.0 | 70.2±1.1 | 77.7±0.2 | 57.8±0.6 | 71.3±1.0 |
| WEMoE + FSFT (full, mean, k=64, seeds=[0, 1, 2]) | 71.6±4.4/80.9±4.3 | 54.0±3.4 | 20.5±17.8 | 89.0±0.6 | 83.1±2.4 | 64.6±3.7 | 79.5±6.3 |
| WEMoE init (full, λ=0.3) | 68.5±0.1/81.3±0.1 | 47.1±0.2